# Causal Inference Statistics Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Causal Inference Statistics**.
It demonstrates core methods for estimating cause-effect relationships from data.

Topics covered:

1. Randomized treatment effect and ATE
2. Confounding bias
3. Propensity score reweighting
4. Difference-in-Differences (DiD)
5. Regression Discontinuity Design (RDD)
6. Instrumental variables
7. Mediation analysis
8. Summary table
9. Mini exercises

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score
np.random.seed(42)

## 1. Randomized Treatment Effect and ATE

The **Average Treatment Effect** (ATE) is the expected causal impact:

$$\text{ATE} = E[Y(1) - Y(0)]$$

With randomized assignment, ATE is unbiasedly estimated by the difference in group means:

$$\hat{\text{ATE}} = \bar{Y}_{\text{treated}} - \bar{Y}_{\text{control}}$$

In [ ]:
n = 400
T    = np.random.binomial(1, 0.5, n)    # random assignment
Y0   = np.random.normal(50, 5, n)        # potential outcome under control
tau_true = 8
Y1   = Y0 + tau_true                     # potential outcome under treatment
Y    = np.where(T == 1, Y1, Y0)          # observed outcome

ate_hat = Y[T == 1].mean() - Y[T == 0].mean()
pd.DataFrame({'Metric': ['True ATE', 'Estimated ATE'], 'Value': [tau_true, ate_hat]})

## 2. Confounding Bias

When treatment assignment depends on a confounder Z that also affects the outcome,
the naive difference in means is biased.

$$\text{Bias} = E[Y|T=1] - E[Y|T=0] - \text{ATE}$$

In [ ]:
Z      = np.random.normal(0, 1, n)
p_treat = 1 / (1 + np.exp(-1.2 * Z))          # treatment depends on Z
T_conf = np.random.binomial(1, p_treat)
Y_conf = 40 + 6*T_conf + 5*Z + np.random.normal(0, 2, n)  # outcome depends on Z too

naive_ate  = Y_conf[T_conf == 1].mean() - Y_conf[T_conf == 0].mean()
true_effect = 6

pd.DataFrame({
    'Metric':  ['True treatment effect', 'Naive (biased) estimate', 'Bias'],
    'Value':   [true_effect, naive_ate, naive_ate - true_effect]
})

## 3. Propensity Score Reweighting

The propensity score e(Z) = P(T=1|Z) can be used to reweight observations and
recover the ATE from observational data:

$$\hat{\text{ATE}} = \frac{1}{n}\sum_{i=1}^n \left(\frac{T_i Y_i}{e(Z_i)} - \frac{(1-T_i)Y_i}{1-e(Z_i)}\right)$$

This is the Inverse Probability Weighting (IPW) estimator.

In [ ]:
ps_model = LogisticRegression().fit(Z.reshape(-1, 1), T_conf)
ps       = ps_model.predict_proba(Z.reshape(-1, 1))[:, 1]

# IPW estimator
ipw_ate = np.mean(T_conf * Y_conf / ps - (1 - T_conf) * Y_conf / (1 - ps))

# Weighted group means
w = np.where(T_conf == 1, 1/ps, 1/(1-ps))
wt_treat   = np.average(Y_conf[T_conf == 1], weights=w[T_conf == 1])
wt_control = np.average(Y_conf[T_conf == 0], weights=w[T_conf == 0])
ate_weighted = wt_treat - wt_control

pd.DataFrame({
    'Metric':  ['Naive ATE', 'IPW ATE', 'Weighted group diff', 'True effect'],
    'Value':   [naive_ate, ipw_ate, ate_weighted, true_effect]
})

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(ps[T_conf == 1], bins=20, alpha=0.6, label='Treated')
plt.hist(ps[T_conf == 0], bins=20, alpha=0.6, label='Control')
plt.title('Propensity Score Distributions')
plt.xlabel('Propensity score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## 4. Difference-in-Differences (DiD)

DiD identifies causal effects by comparing **before-after changes** across a treated and a control group.

$$\hat{\text{ATE}}_{DiD} = (\bar{Y}_{\text{treat},post} - \bar{Y}_{\text{treat},pre}) - (\bar{Y}_{\text{control},post} - \bar{Y}_{\text{control},pre})$$

**Key assumption:** Parallel trends — without treatment, both groups would have changed by the same amount.

This is widely used in policy evaluation, A/B testing with pre-periods, and natural experiments.

In [ ]:
n_did = 200
true_did_effect = 10.0

# Baseline outcomes (pre-treatment)
Y_treat_pre    = np.random.normal(50, 5, n_did)
Y_control_pre  = np.random.normal(48, 5, n_did)   # different baseline OK for DiD

# Post outcomes: both groups trend up by 5 (parallel trend), treated gets extra effect
common_trend   = 5.0
Y_treat_post   = Y_treat_pre   + common_trend + true_did_effect + np.random.normal(0, 2, n_did)
Y_control_post = Y_control_pre + common_trend                   + np.random.normal(0, 2, n_did)

did_estimate = ((Y_treat_post.mean() - Y_treat_pre.mean()) -
                (Y_control_post.mean() - Y_control_pre.mean()))

# Regression-based DiD
post_indicator  = np.concatenate([np.ones(n_did), np.zeros(n_did),
                                   np.ones(n_did), np.zeros(n_did)])
treat_indicator = np.concatenate([np.ones(n_did),  np.ones(n_did),
                                   np.zeros(n_did), np.zeros(n_did)])
Y_all           = np.concatenate([Y_treat_post, Y_treat_pre, Y_control_post, Y_control_pre])
interaction     = post_indicator * treat_indicator
X_did           = np.column_stack([np.ones(len(Y_all)), treat_indicator, post_indicator, interaction])
beta_did        = np.linalg.lstsq(X_did, Y_all, rcond=None)[0]

pd.DataFrame({
    'Metric':   ['True DiD effect', 'Simple DiD estimate', 'Regression DiD coeff (interaction)'],
    'Value':    [true_did_effect, did_estimate, beta_did[3]]
})

In [ ]:
means_df = pd.DataFrame({
    'Group':        ['Treated', 'Treated', 'Control', 'Control'],
    'Period':       ['Pre', 'Post', 'Pre', 'Post'],
    'Mean outcome': [Y_treat_pre.mean(), Y_treat_post.mean(),
                     Y_control_pre.mean(), Y_control_post.mean()]
})

plt.figure(figsize=(6, 4))
for grp, color in [('Treated', 'tab:blue'), ('Control', 'tab:orange')]:
    sub = means_df[means_df['Group'] == grp]
    plt.plot(['Pre', 'Post'], sub['Mean outcome'].values, marker='o', label=grp, color=color)
plt.title('Difference-in-Differences: Parallel Trends')
plt.ylabel('Mean outcome')
plt.legend()
plt.show()

## 5. Regression Discontinuity Design (RDD)

RDD exploits a **threshold** in a running variable that determines treatment assignment.
Units just below and just above the cutoff are assumed comparable, so the jump at the
threshold estimates the local causal effect.

$$\hat{\text{LATE}} = \lim_{x \to c^+} E[Y|X=x] - \lim_{x \to c^-} E[Y|X=x]$$

where c is the cutoff and LATE is the Local Average Treatment Effect at the threshold.

In [ ]:
n_rdd = 400
cutoff = 50.0
true_rdd_effect = 12.0

running = np.random.uniform(20, 80, n_rdd)          # running variable (e.g., test score)
treated_rdd = (running >= cutoff).astype(int)

# Outcome: smooth function of running + jump at cutoff
Y_rdd = (0.5 * running + true_rdd_effect * treated_rdd +
         np.random.normal(0, 3, n_rdd))

# Local linear regression near the cutoff (bandwidth = 10)
bw = 10
mask = np.abs(running - cutoff) <= bw

X_rdd_local = np.column_stack([
    np.ones(mask.sum()),
    running[mask] - cutoff,           # centered running variable
    treated_rdd[mask],
    (running[mask] - cutoff) * treated_rdd[mask]  # interaction
])
beta_rdd = np.linalg.lstsq(X_rdd_local, Y_rdd[mask], rcond=None)[0]
rdd_estimate = beta_rdd[2]   # coefficient on treatment indicator

pd.DataFrame({
    'Metric':   ['True RDD effect', 'Local linear estimate', 'Bandwidth', 'Obs near cutoff'],
    'Value':    [true_rdd_effect, rdd_estimate, bw, mask.sum()]
})

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(running[treated_rdd == 0], Y_rdd[treated_rdd == 0], alpha=0.3, s=10, label='Control')
plt.scatter(running[treated_rdd == 1], Y_rdd[treated_rdd == 1], alpha=0.3, s=10, label='Treated')
plt.axvline(cutoff, linestyle='--', color='black', label='Cutoff')
plt.title('Regression Discontinuity Design')
plt.xlabel('Running variable')
plt.ylabel('Outcome')
plt.legend()
plt.show()

## 6. Instrumental Variables (IV)

An instrument Z is a variable that:
1. Affects the treatment X (relevance)
2. Does not affect the outcome Y except through X (exclusion restriction)

The Wald IV estimator:

$$\hat{\beta}_{IV} = \frac{\text{Cov}(Z, Y)}{\text{Cov}(Z, X)}$$

In [ ]:
Z_iv  = np.random.binomial(1, 0.5, n)
X_iv  = 0.7 * Z_iv + 0.8 * np.random.normal(0, 1, n)   # X partly determined by Z
Y_iv  = 3 * X_iv + np.random.normal(0, 1, n)             # true effect of X on Y is 3

wald = np.cov(Z_iv, Y_iv, bias=True)[0, 1] / np.cov(Z_iv, X_iv, bias=True)[0, 1]

pd.DataFrame({'Metric': ['True effect of X', 'Wald IV estimate'], 'Value': [3, wald]})

## 7. Mediation Analysis

Mediation analysis decomposes a total effect into:
- **Direct effect:** X affects Y directly
- **Indirect (mediated) effect:** X affects M which affects Y

Total effect = Direct effect + Indirect effect

In [ ]:
X_med = np.random.binomial(1, 0.5, n)
M     = 2 * X_med + np.random.normal(0, 1, n)
Y_med = 1.5 * X_med + 3 * M + np.random.normal(0, 1, n)

# Total effect: regress Y on X only
model_total  = LinearRegression().fit(X_med.reshape(-1, 1), Y_med)

# Direct effect: regress Y on both X and M
model_direct = LinearRegression().fit(np.column_stack([X_med, M]), Y_med)

direct_effect   = model_direct.coef_[0]
mediator_effect = model_direct.coef_[1]
indirect_effect = model_total.coef_[0] - direct_effect

pd.DataFrame({
    'Effect':  ['Total effect of X', 'Direct effect of X', 'Indirect (via M)', 'Effect of M on Y'],
    'Estimate':[model_total.coef_[0], direct_effect, indirect_effect, mediator_effect]
})

## 8. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Measure':   ['Randomized ATE', 'Naive confounded ATE', 'IPW ATE',
                  'DiD estimate', 'RDD estimate', 'IV estimate', 'Direct effect (mediation)'],
    'Value':     [ate_hat, naive_ate, ipw_ate,
                  did_estimate, rdd_estimate, wald, direct_effect]
})
summary

## 9. Mini Exercises

Try these on your own:

1. Change the true treatment effect to 15 in the randomized experiment and verify the estimate recovers it.
2. Increase the confounding strength (coefficient on Z in the treatment model) and observe how bias grows.
3. In the DiD example, violate the parallel trends assumption by adding group-specific trends and measure the resulting bias.
4. Narrow the RDD bandwidth from 10 to 5 and compare the estimate with more vs fewer observations.
5. Create a scenario where IV is weak (low correlation between Z and X) and observe what happens to the Wald estimate.
6. Increase the mediator effect and verify that the indirect effect grows accordingly in the mediation example.
7. Combine propensity score reweighting with a regression adjustment and compare to pure IPW.
8. Apply DiD to a two-group dataset from an A/B test with a pre-period measurement.

These exercises are especially useful for policy evaluation, A/B testing, econometrics, medical research, and AI fairness.